# Benchmark Pipeline Lab

Notebook nay kiem tra pipeline benchmark cua du an, doi chieu `VNINDEX` that voi `market_proxy`, va giai thich benchmark source dang duoc screener/backtest su dung.

- Neu co `data/processed/benchmark_vnindex.parquet`: uu tien dung benchmark that
- Neu chua co: he thong se fallback sang `market_proxy` va notebook nay se chi ro dieu do


In [1]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path

def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")

project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [2]:
# Cell: Import Benchmark Modules
import importlib
import inspect

import pandas as pd

import src.analytics.benchmark as benchmark_analytics
import src.reporting.chart_factory as chart_factory
from src.data.loaders import read_parquet

benchmark_analytics = importlib.reload(benchmark_analytics)
chart_factory = importlib.reload(chart_factory)

load_processed_benchmark = benchmark_analytics.load_processed_benchmark
build_market_proxy_benchmark = benchmark_analytics.build_market_proxy_benchmark
compare_benchmark_to_proxy = benchmark_analytics.compare_benchmark_to_proxy
summarize_benchmark_alignment = benchmark_analytics.summarize_benchmark_alignment
benchmark_nav_chart = chart_factory.benchmark_nav_chart
benchmark_spread_chart = chart_factory.benchmark_spread_chart

print(inspect.signature(compare_benchmark_to_proxy))


(price_df: 'pd.DataFrame', benchmark_df: 'pd.DataFrame | None' = None, benchmark_path: 'str | Path' = 'data/processed/benchmark_vnindex.parquet', benchmark_symbol: 'str' = 'VNINDEX', date_col: 'str' = 'date', return_col: 'str' = 'return_1d', price_col: 'str' = 'close', window: 'int' = 20) -> 'tuple[pd.DataFrame, str]'


In [3]:
# Cell: Load Benchmark Inputs
price_df = read_parquet("data/processed/clean_daily_ohlcv.parquet")
benchmark_df = read_parquet("data/processed/benchmark_vnindex.parquet")
screen_backtest_summary = read_parquet("data/signals/stealth_accumulation_backtest_summary.parquet")
screen_backtest_by_date = read_parquet("data/signals/stealth_accumulation_backtest_by_date.parquet")

coverage = {
    "stock_symbols": int(price_df["symbol"].nunique()) if not price_df.empty else 0,
    "price_date_min": str(pd.to_datetime(price_df["date"]).min().date()) if not price_df.empty else None,
    "price_date_max": str(pd.to_datetime(price_df["date"]).max().date()) if not price_df.empty else None,
    "benchmark_rows": int(len(benchmark_df)),
    "benchmark_available": bool(not benchmark_df.empty),
}
pd.DataFrame([coverage]).T.rename(columns={0: "value"})


,value
stock_symbols,500
price_date_min,2020-01-02
price_date_max,2026-07-20
benchmark_rows,1632
benchmark_available,True


In [4]:
# Cell: Build Benchmark Comparison
benchmark_comparison, benchmark_source = compare_benchmark_to_proxy(
    price_df=price_df,
    benchmark_df=benchmark_df,
    benchmark_path="data/processed/benchmark_vnindex.parquet",
    benchmark_symbol="VNINDEX",
)

benchmark_comparison[[
    "date",
    "market_proxy_return_1d",
    "benchmark_return_1d",
    "return_spread",
    "rolling_correlation_20d",
    "benchmark_source",
]].tail(10)


,date,market_proxy_return_1d,benchmark_return_1d,return_spread,rolling_correlation_20d,benchmark_source
1622,2026-07-07,0.003039,0.260000,0.256961,0.156821,benchmark_df
1623,2026-07-08,0.002902,0.290000,0.287098,0.221285,benchmark_df
1624,2026-07-09,-0.003678,-0.007013,-0.003335,0.248211,benchmark_df
1625,2026-07-10,-0.005392,-0.006715,-0.001323,0.253304,benchmark_df
1626,2026-07-13,-0.016165,-0.015205,0.000959,0.278950,benchmark_df
1627,2026-07-14,0.002781,0.003382,0.000602,0.296528,benchmark_df
1628,2026-07-15,-0.010491,-0.013567,-0.003076,0.321948,benchmark_df
1629,2026-07-16,0.000855,0.012412,0.011558,0.315705,benchmark_df
1630,2026-07-17,-0.004066,-0.009306,-0.005240,0.317083,benchmark_df
1631,2026-07-20,-0.023462,-0.024583,-0.001121,0.304945,benchmark_df


In [5]:
# Cell: Review Benchmark Summary
benchmark_summary = summarize_benchmark_alignment(benchmark_comparison)
pd.DataFrame([benchmark_summary]).T.rename(columns={0: "value"})


,value
rows,1632
date_min,2020-01-02
date_max,2026-07-20
benchmark_source,benchmark_df
coverage_ratio,1.0
mean_return_spread,0.003895
tracking_error_20d,1.845356
rolling_correlation_20d,0.304945


In [6]:
# Cell: Visualize Benchmark Alignment
display(benchmark_nav_chart(benchmark_comparison, title="Benchmark NAV Vs Market Proxy NAV"))
display(benchmark_spread_chart(benchmark_comparison, title="Benchmark Minus Market Proxy Spread"))


In [7]:
# Cell: Review Screener Backtest Context
if screen_backtest_summary.empty:
    display(pd.DataFrame({"message": ["No screener backtest summary found. Run notebook 14 or the screen script first."]}))
else:
    display(screen_backtest_summary)

if not screen_backtest_by_date.empty:
    display(screen_backtest_by_date[[
        "selection_date",
        "holding_period_days",
        "portfolio_type",
        "avg_forward_return",
        "benchmark_return",
        "excess_return",
        "beat_benchmark",
        "benchmark_source",
    ]].tail(20))


,portfolio_type,holding_period_days,observations,avg_forward_return,avg_net_forward_return,avg_trading_cost,avg_signal_to_entry_gap,avg_benchmark_return,avg_excess_return,avg_net_excess_return,...,excess_hit_rate,net_excess_hit_rate,avg_selection_count,volatility_of_avg_returns,volatility_of_avg_net_returns,sharpe_like,net_sharpe_like,information_ratio_like,net_information_ratio_like,benchmark_source
0,raw,5,1018,0.001348,-0.004758,0.006106,0.000411,0.001028,0.000319,-0.005787,...,0.486248,0.385069,4.350688,0.034582,0.034557,0.276655,-0.977558,0.079851,-1.446441,benchmark_vnindex.parquet
1,raw,10,1013,0.003486,-0.002622,0.006108,0.000436,0.003133,0.000353,-0.005755,...,0.491609,0.421520,4.347483,0.048401,0.048382,0.361577,-0.272047,0.042672,-0.694624,benchmark_vnindex.parquet
2,raw,20,1003,0.006827,0.000717,0.006110,0.000441,0.006023,0.000805,-0.005305,...,0.480558,0.421735,4.340977,0.069085,0.069093,0.350796,0.036853,0.051315,-0.338229,benchmark_vnindex.parquet
3,sector_aware,5,1018,0.001650,-0.004385,0.006035,0.000281,0.001028,0.000622,-0.005413,...,0.501965,0.382122,3.269155,0.032698,0.032663,0.358335,-0.953061,0.154059,-1.340343,benchmark_vnindex.parquet
4,sector_aware,10,1013,0.003416,-0.002621,0.006037,0.000308,0.003133,0.000283,-0.005754,...,0.508391,0.422507,3.270484,0.046000,0.045986,0.372750,-0.286149,0.033957,-0.690322,benchmark_vnindex.parquet
5,sector_aware,20,1003,0.003458,-0.002583,0.006041,0.000317,0.006023,-0.002564,-0.008605,...,0.457627,0.415753,3.283151,0.066675,0.066695,0.184118,-0.137449,-0.160811,-0.539198,benchmark_vnindex.parquet


,selection_date,holding_period_days,portfolio_type,avg_forward_return,benchmark_return,excess_return,beat_benchmark,benchmark_source
6048,2026-05-26,20,sector_aware,-0.046568,-0.008035,-0.038533,False,benchmark_vnindex.parquet
6049,2026-05-27,20,sector_aware,-0.048652,0.001915,-0.050568,False,benchmark_vnindex.parquet
6050,2026-05-28,20,sector_aware,-0.028143,-0.000322,-0.027821,False,benchmark_vnindex.parquet
6051,2026-05-29,20,sector_aware,-0.037532,0.004518,-0.042051,False,benchmark_vnindex.parquet
6052,2026-06-01,20,sector_aware,-0.029193,0.005655,-0.034848,False,benchmark_vnindex.parquet
6053,2026-06-02,20,sector_aware,-0.036957,0.018363,-0.055320,False,benchmark_vnindex.parquet
6054,2026-06-03,20,sector_aware,0.023694,0.026498,-0.002804,False,benchmark_vnindex.parquet
6055,2026-06-04,20,sector_aware,0.003477,0.019000,-0.015523,False,benchmark_vnindex.parquet
6056,2026-06-05,20,sector_aware,-0.018431,0.012605,-0.031037,False,benchmark_vnindex.parquet
6057,2026-06-08,20,sector_aware,-0.019570,0.029583,-0.049153,False,benchmark_vnindex.parquet


In [8]:
# Cell: Diagnose Next Step
if benchmark_source == "market_proxy_only":
    print("Benchmark that chua san sang. Can nap du lieu VNINDEX vao data/raw/index_ohlcv roi chay lai generate_features.py")
else:
    print(f"Dang dung benchmark source: {benchmark_source}")


Dang dung benchmark source: benchmark_df
